## Notebook 4: Matching Pipeline & RAG

### This Notebook
This notebook builds the full end-to-end grant matching pipeline by combining
the semantic search index from Notebook 3 with an LLM explanation layer.

The pipeline has two stages:

**Stage 1 — Semantic Retrieval:** The org's plain-language description is
embedded using the same sentence transformer model used to index the grants.
FAISS finds the k most semantically similar grant programs in milliseconds.

**Stage 2 — LLM Analysis:** The top matches are passed to Llama 3.1
(via Groq) with a structured prompt that asks it to explain fit, flag
eligibility concerns, and prioritize the best opportunities for that
specific org type and location.

**Core functions exported to Notebook 5:**
- `search_grants(query, k)` — semantic retrieval only
- `explain_matches(org_description, results, org_type, org_state)` — LLM layer only  
- `find_grants(org_description, org_type, org_state, k)` — full pipeline

**Inputs:** `grant_profiles.csv`, `grant_embeddings.npy`, `grant_index.faiss` (from Notebooks 2 & 3)

In [20]:
!pip install faiss-cpu # Install faiss-cpu
!pip install groq -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.3 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from google.colab import drive
import anthropic

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/grant_finder'

# Load everything built in previous notebooks
grant_profiles = pd.read_csv(f'{DRIVE_PATH}/grant_profiles.csv')
grant_embeddings = np.load(f'{DRIVE_PATH}/grant_embeddings.npy')
index = faiss.read_index(f'{DRIVE_PATH}/grant_index.faiss')
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Grant profiles: {len(grant_profiles):,}')
print(f'Embeddings shape: {grant_embeddings.shape}')
print(f'FAISS index vectors: {index.ntotal}')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Grant profiles: 1,222
Embeddings shape: (1222, 384)
FAISS index vectors: 1222


In [22]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Test
response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': 'Say "grant_finder is ready" and nothing else.'}]
)

grant_finder is ready


# RAG Pipeline

In [29]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Sentence transformer loaded')

def search_grants(query, k=10):
    """Search for matching grant programs given an org description."""
    query_vector = st_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, k=k)
    results = grant_profiles.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]
    return results[[
        'cfda_number', 'cfda_title', 'awarding_agency',
        'num_recipients', 'avg_award', 'min_award', 'max_award',
        'org_types', 'embedding_text', 'similarity_score'
    ]].reset_index(drop=True)

def explain_matches(org_description, search_results, org_type=None, org_state=None):
    """
    Pass top semantic matches to LLM for explanation and eligibility analysis.

    Args:
        org_description: plain language description of the org
        search_results: DataFrame from search_grants()
        org_type: optional - 'Nonprofit', 'University / Higher Education', 'Small Business'
        org_state: optional - two letter state code

    Returns:
        LLM response string
    """
    # Build context string from top 5 matches
    grants_context = ""
    for i, row in search_results.head(5).iterrows():
        grants_context += f"""
---
Grant Program {i+1}:
CFDA: {row['cfda_number']}
Title: {row['cfda_title']}
Agency: {row['awarding_agency']}
Avg Award: ${row['avg_award']:,.0f}
Award Range: ${row['min_award']:,.0f} - ${row['max_award']:,.0f}
Typically funds: {row['org_types']}
Similarity score: {row['similarity_score']:.3f}
Program description: {row['embedding_text'][:500]}
"""

    org_context = f"Organization description: {org_description}"
    if org_type:
        org_context += f"\nOrganization type: {org_type}"
    if org_state:
        org_context += f"\nState: {org_state}"

    prompt = f"""You are a federal grant matching assistant. Given an organization's description
and a list of potentially matching federal grant programs, provide a clear analysis of the best matches.

{org_context}

Here are the top semantically matching grant programs from the USASpending database:
{grants_context}

For each grant program, provide:
1. Why it is or isn't a good fit for this organization
2. Any eligibility concerns based on the org type and program description
3. A recommended priority (High / Medium / Low) for pursuing this grant

Be specific and practical. Focus on the top 3 most promising matches."""

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content


def find_grants(org_description, org_type=None, org_state=None, k=10):
    """
    Full pipeline: semantic search + LLM explanation.
    Main entry point for Notebook 5.

    Args:
        org_description: plain language description of the org
        org_type: optional - 'Nonprofit', 'University / Higher Education', 'Small Business'
        org_state: optional - two letter state code
        k: number of semantic matches to retrieve before LLM ranking

    Returns:
        search_results: DataFrame of top k matches
        explanation: LLM explanation string
    """
    print('Searching for matching grants...')
    search_results = search_grants(org_description, k=k)

    print('Generating explanation...')
    explanation = explain_matches(org_description, search_results, org_type, org_state)

    return search_results, explanation

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence transformer loaded


In [30]:
# Full pipeline test
org_desc = "We are a nonprofit providing mental health counseling and substance abuse treatment in rural communities across Appalachia"

results, explanation = find_grants(
    org_description=org_desc,
    org_type='Nonprofit',
    org_state='WV',
    k=10
)

print('=== TOP SEMANTIC MATCHES ===')
print(results[['cfda_number', 'cfda_title', 'awarding_agency', 'avg_award', 'similarity_score']].to_string())
print('\n=== LLM ANALYSIS ===')
print(explanation)

Searching for matching grants...
Generating explanation...
=== TOP SEMANTIC MATCHES ===
   cfda_number                                                                                 cfda_title                          awarding_agency     avg_award  similarity_score
0       93.958                                          BLOCK GRANTS FOR COMMUNITY MENTAL HEALTH SERVICES  Department of Health and Human Services  4.091098e+05          0.613886
1       93.087                                     ENHANCE SAFETY OF CHILDREN AFFECTED BY SUBSTANCE ABUSE  Department of Health and Human Services  5.954761e+05          0.605265
2       84.184                                                          SCHOOL SAFELY NATIONAL ACTIVITIES                  Department of Education  5.414552e+05          0.596564
3       93.788                                                                                 OPIOID STR  Department of Health and Human Services  4.463162e+05          0.590366
4       93.732   

# Test # 2

In [31]:
# Test with a university
results2, explanation2 = find_grants(
    org_description="We are a public university conducting materials science and engineering research focused on renewable energy applications and battery technology",
    org_type='University / Higher Education',
    org_state='OH',
    k=10
)
print('=== LLM ANALYSIS — UNIVERSITY ===')
print(explanation2)

Searching for matching grants...
Generating explanation...
=== LLM ANALYSIS — UNIVERSITY ===
Based on the organization's description and the list of potentially matching federal grant programs, I will provide an analysis of the top 3 most promising matches.

**Grant Program 1: MANUFACTURING AND ENERGY SUPPLY CHAIN DEMONSTRATIONS AND COMMERCIAL APPLICATIONS (CFDA: 81.253)**

1. Why it is a good fit: This program aligns closely with the organization's focus on materials science and engineering research for renewable energy applications and battery technology. The program's emphasis on manufacturing and energy supply chain demonstrations also matches the organization's research goals.
2. Eligibility concerns: The program typically funds for-profit organizations and small businesses, which may not be a good fit for a public university. However, the program description does not explicitly exclude universities, so it's worth exploring further.
3. Recommended priority: Medium

**Grant Program

Test # 3

In [32]:
# Test with a small business
results3, explanation3 = find_grants(
    org_description="We are a small biotech company developing novel diagnostic tools for early cancer detection using machine learning and genomic data",
    org_type='Small Business',
    org_state='MA',
    k=10
)
print('=== LLM ANALYSIS — SMALL BUSINESS ===')
print(explanation3)

Searching for matching grants...
Generating explanation...
=== LLM ANALYSIS — SMALL BUSINESS ===
Based on the organization's description and the list of potentially matching federal grant programs, I've analyzed each program and provided the requested information.

**Grant Program 1: PROMOTING THE CANCER SURVEILLANCE WORKFORCE, EDUCATION AND DATA USE (CFDA: 93.832)**

1. Why it is or isn't a good fit: This program is a good fit because it focuses on cancer surveillance, education, and data use, which aligns with the organization's mission of developing novel diagnostic tools for early cancer detection. However, the program description emphasizes workforce development and education, which might not directly align with the organization's primary focus on tool development.
2. Eligibility concerns: The program typically funds non-profit organizations without 501(c)(3) status, which is not a concern for this organization. However, the program description mentions a focus on workforce develo